In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score
    )
from sklearn.utils.class_weight import compute_class_weight

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:

# ---------- Config ----------
DATA_PATH = '/kaggle/input/datasets/iammustafatz/diabetes-prediction-dataset/diabetes_prediction_dataset.csv'
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
BATCH_SIZE = 256
MAX_EPOCHS = 80
PATIENCE = 10

# Recall-focused settings
USE_WEIGHTED_SAMPLER = False
USE_POS_WEIGHT = False

# Threshold policy options: 'f1', 'precision_at_min_recall', 'recall_at_min_precision'
THRESHOLD_MODE = 'recall_at_min_precision'
MIN_RECALL_TARGET = 0.45
MIN_PRECISION_TARGET = 0.70

# Wider threshold sweep helps when best threshold is near the upper edge
THRESHOLD_GRID_MIN = 0.05
THRESHOLD_GRID_MAX = 0.999
THRESHOLD_GRID_STEPS = 500


In [ ]:

# Load data 
df = pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape)
print(df.dtypes)
print(df.isnull().sum())

# Keep only Female/Male because Other is extremely rare
df = df[df['gender'].isin(['Female', 'Male'])].reset_index(drop=True)

# Encode gender
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1}).astype(int)

# Encode smoking history as an ordinal risk feature
smoking_map = {
    'never': 0,
    'No Info': 1,
    'former': 2,
    'ever': 2,
    'not current': 2,
    'current': 3
}
df['smoking_history'] = df['smoking_history'].map(smoking_map).fillna(1).astype(int)

feature_cols = [
    'gender',
    'age',
    'hypertension',
    'heart_disease',
    'smoking_history',
    'bmi',
    'HbA1c_level',
    'blood_glucose_level'
 ]
target_col = 'diabetes'

X = df[feature_cols].copy()
y = df[target_col].astype(int).values

print('\nFeature columns:', feature_cols)
print('X shape:', X.shape)
print('Class balance:')
print(pd.Series(y).value_counts(normalize=True))


In [ ]:

# Split data 
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'\nTrain size: {len(y_train)}, Val size: {len(y_val)}, Test size: {len(y_test)}')


In [ ]:
# Dataset
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = DiabetesDataset(X_train, y_train)
val_dataset = DiabetesDataset(X_val, y_val)
test_dataset = DiabetesDataset(X_test, y_test)

if USE_WEIGHTED_SAMPLER:
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    sample_weights = [class_weights[int(label)] for label in y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    print('Train loader: weighted sampler')
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print('Train loader: standard shuffle (no weighted sampler)')

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:

#  Model
class MonotonicDiabetesNet(nn.Module):
    def __init__(self, input_dim, monotonic_feature_idx):
        super().__init__()
        self.monotonic_feature_idx = monotonic_feature_idx
        self.base_feature_idx = [i for i in range(input_dim) if i not in monotonic_feature_idx]

        self.base_net = nn.Sequential(
            nn.Linear(len(self.base_feature_idx), 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # Constrain these weights to be >= 0 via softplus => monotonic non-decreasing effect
        self.raw_monotonic_weights = nn.Parameter(torch.zeros(len(monotonic_feature_idx)))
        self.monotonic_bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        x_base = x[:, self.base_feature_idx]
        x_mono = x[:, self.monotonic_feature_idx]

        base_logit = self.base_net(x_base)
        mono_weights = torch.nn.functional.softplus(self.raw_monotonic_weights).unsqueeze(1)
        mono_logit = x_mono @ mono_weights

        return base_logit + mono_logit + self.monotonic_bias

MONOTONIC_FEATURES = ['HbA1c_level', 'blood_glucose_level']
monotonic_idx = [feature_cols.index(col) for col in MONOTONIC_FEATURES]
print('Monotonic features:', MONOTONIC_FEATURES, '| idx:', monotonic_idx)

model = MonotonicDiabetesNet(input_dim=X_train.shape[1], monotonic_feature_idx=monotonic_idx).to(device)


In [ ]:

# Loss / optimizer 
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
pos_weight_value = float(np.clip(neg_count / max(pos_count, 1), 1.0, 8.0))

if USE_POS_WEIGHT:
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
    )
    print(f'Loss: BCEWithLogits with pos_weight={pos_weight_value:.3f} (neg={neg_count}, pos={pos_count})')
else:
    criterion = nn.BCEWithLogitsLoss()
    print('Loss: BCEWithLogits without pos_weight (precision-focused)')

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)


In [ ]:

# Train / eval
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            probs = torch.sigmoid(logits)
            total_loss += criterion(logits, y_b).item()
            all_probs.extend(probs.cpu().numpy().ravel())
            all_labels.extend(y_b.cpu().numpy().ravel())
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels).astype(int)
    roc_auc = roc_auc_score(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)
    return total_loss / len(loader), all_labels, all_probs, roc_auc, pr_auc

def select_threshold(
    labels,
    probs,
    mode='f1',
    min_recall=0.55,
    min_precision=0.45,
    thr_min=0.05,
    thr_max=0.999,
    thr_steps=500
    ):
    thresholds = np.linspace(thr_min, thr_max, thr_steps)
    rows = []
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        rows.append({
            'thr': thr,
            'precision': precision_score(labels, preds, zero_division=0),
            'recall': recall_score(labels, preds, zero_division=0),
            'f1': f1_score(labels, preds, zero_division=0)
        })
    metric_df = pd.DataFrame(rows)
    if mode == 'precision_at_min_recall':
        candidates = metric_df[metric_df['recall'] >= min_recall]
        if len(candidates) > 0:
            best = candidates.sort_values(['precision', 'f1', 'thr'], ascending=[False, False, False]).iloc[0]
        else:
            best = metric_df.sort_values(['recall', 'f1'], ascending=[False, False]).iloc[0]
    elif mode == 'recall_at_min_precision':
        candidates = metric_df[metric_df['precision'] >= min_precision]
        if len(candidates) > 0:
            best = candidates.sort_values(['recall', 'f1'], ascending=[False, False]).iloc[0]
        else:
            best = metric_df.sort_values(['precision', 'f1'], ascending=[False, False]).iloc[0]
    else:
        best = metric_df.sort_values('f1', ascending=False).iloc[0]
    return float(best['thr']), metric_df




In [ ]:
# Train
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(MAX_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_labels, val_probs, val_roc_auc, val_pr_auc = eval_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stop at epoch {epoch + 1}')
            break

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch + 1:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_roc_auc:.4f} | Val PR-AUC: {val_pr_auc:.4f}')

model.load_state_dict(best_state)
model.eval()

In [ ]:

# Final validation-based threshold and test evaluation 
_, val_labels, val_probs, val_roc_auc, val_pr_auc = eval_epoch(model, val_loader, criterion)
best_thr, threshold_grid = select_threshold(
    val_labels,
    val_probs,
    mode=THRESHOLD_MODE,
    min_recall=MIN_RECALL_TARGET,
    min_precision=MIN_PRECISION_TARGET,
    thr_min=THRESHOLD_GRID_MIN,
    thr_max=THRESHOLD_GRID_MAX,
    thr_steps=THRESHOLD_GRID_STEPS
)

_, test_labels, test_probs, test_roc_auc, test_pr_auc = eval_epoch(model, test_loader, criterion)
test_preds = (test_probs >= best_thr).astype(int)

test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)

print(f'\nValidation ROC-AUC: {val_roc_auc:.4f}')
print(f'Validation PR-AUC : {val_pr_auc:.4f}')
if THRESHOLD_MODE == 'precision_at_min_recall':
    print(f'Threshold policy: maximize precision with recall >= {MIN_RECALL_TARGET:.2f}')
elif THRESHOLD_MODE == 'recall_at_min_precision':
    print(f'Threshold policy: maximize recall with precision >= {MIN_PRECISION_TARGET:.2f}')
else:
    print('Threshold policy: maximize F1')
print(f'Chosen threshold from validation: {best_thr:.3f}\n')

print(f'Test ROC-AUC: {test_roc_auc:.4f}')
print(f'Test PR-AUC : {test_pr_auc:.4f}\n')
print(classification_report(test_labels, test_preds, target_names=['No Diabetes', 'Diabetes'], zero_division=0))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Test)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
sns.lineplot(data=threshold_grid, x='thr', y='precision', label='precision')
sns.lineplot(data=threshold_grid, x='thr', y='recall', label='recall')
sns.lineplot(data=threshold_grid, x='thr', y='f1', label='f1')
plt.axvline(best_thr, color='red', linestyle='--', label=f'best thr={best_thr:.3f}')
plt.title('Validation metrics vs threshold')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import os
import json
import joblib
from IPython.display import FileLink, display

# Create output folder in Kaggle working directory
save_dir = "/kaggle/working/mustafa_model"
os.makedirs(save_dir, exist_ok=True)

# 1) Save PyTorch weights
torch.save(model.state_dict(), f"{save_dir}/mustafa_model.pth")

# 2) Save preprocessing objects and metadata
joblib.dump(scaler, f"{save_dir}/scaler.joblib")

metadata = {
    "feature_cols": feature_cols,
    "threshold": float(best_thr),
    "threshold_mode": THRESHOLD_MODE,
    "min_recall_target": float(MIN_RECALL_TARGET),
    "min_precision_target": float(MIN_PRECISION_TARGET),
    "model_class": "DiabetesNet",
    "input_dim": int(X_train.shape[1]),
}
with open(f"{save_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

import shutil
zip_path = "/kaggle/working/mustafa_model.zip"
shutil.make_archive("/kaggle/working/mustafa_model", "zip", save_dir)

print("Saved files:")
print(f"{save_dir}/mustafa_model.pth")
print(f"{save_dir}/scaler.joblib")
print(f"{save_dir}/metadata.json")
print(zip_path)

display(FileLink(zip_path))

In [ ]:
Validation ROC-AUC: 0.9780
Validation PR-AUC : 0.8822
Threshold policy: maximize precision with recall >= 0.55
Chosen threshold from validation: 0.95

Test ROC-AUC: 0.9768
Test PR-AUC : 0.8770

              precision    recall  f1-score   support

 No Diabetes       0.98      0.98      0.98     18297
    Diabetes       0.75      0.78      0.76      1700

    accuracy                           0.96     19997
   macro avg       0.87      0.88      0.87     19997
weighted avg       0.96      0.96      0.96     19997


